# Full ColorFeret cGAN Training
This notebook orchestrates end-to-end training on the complete 11,338 image ColorFeret dataset.
It uses the pre-uploaded Kaggle dataset and linked pretrained weights securely to train the GitHub repository on Kaggle.

## 1. Check Kaggle GPU & Setup Environment

In [ ]:
import os
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone the GitHub Repository

In [ ]:
!rm -rf stylized-portrait-generation
!git clone https://github.com/supratimcoder1/stylized-portrait-generation.git
%cd stylized-portrait-generation

## 3. Install Dependencies

In [ ]:
%pip install -r requirements.txt -q

## 4. Verify Split Dataset & Weights Location

In [ ]:
# Assumes Kaggle Dataset names: color-feret-splits and best-model-pth
base_dir = "/kaggle/input/color-feret-splits/full_dataset"
splits = ["train_images", "train_targets", "val_images", "val_targets"]
weights_path = "/kaggle/input/best-model-pth/best_model.pth"

print(f"Using split dataset base: {base_dir}")
for split in splits:
    path = os.path.join(base_dir, split)
    exists = os.path.exists(path)
    count = len(os.listdir(path)) if exists else 0
    print(f"{split} exists: {exists} | files: {count}")
print(f"Pretrained weights exist: {os.path.exists(weights_path)}")

## 5. Create Split Symlinks and Model Setup
Link read-only Kaggle split folders into the repo's expected dataset/ layout.

In [ ]:
!rm -rf dataset
!mkdir -p dataset
!mkdir -p checkpoints

# Symlink split folders expected by training/train.py
!ln -s "$base_dir/train_images" dataset/train_images
!ln -s "$base_dir/train_targets" dataset/train_targets
!ln -s "$base_dir/val_images" dataset/val_images
!ln -s "$base_dir/val_targets" dataset/val_targets
!ln -s {weights_path} checkpoints/best_model.pth

print("Linked dataset layout:")
!ls -la dataset/
print("\nLinked checkpoints layout:")
!ls -la checkpoints/

## 6. Start Training

In [ ]:
!python training/train.py --epochs 100 --batch-size 8 --output-dir /kaggle/working/training_outputs --checkpoint-dir ./checkpoints

## 7. Inspect Training Outputs

In [ ]:
import glob
from IPython.display import display, Image as IPImage

sample_files = sorted(glob.glob("/kaggle/working/training_outputs/epoch_*.png"))
if sample_files:
    latest = sample_files[-1]
    print(f"Showing: {latest}")
    display(IPImage(filename=latest, width=800))
else:
    print("No sample images found - training may not have completed.")

## 8. Download Best Trained Model

In [ ]:
model_path = "./checkpoints/final_model.pth"
if os.path.exists(model_path):
    !zip -j /kaggle/working/final_model.zip {model_path}
    print("Best model zipped to /kaggle/working/final_model.zip")
else:
    print("final_model.pth not found. Check training logs and checkpoint directory.")